In [1]:
import pandas as pd
import numpy as np
import netCDF4 as nc
from pathlib import Path
import re
from datetime import datetime
import glob
from collections import defaultdict

def batch_csv_to_netcdf(data_dir, output_filename='greenland_albedo_timeseries.nc', pattern='*Greenland_albedo.csv'):

    
    data_dir = Path(data_dir)
    output_path = data_dir / output_filename
    
    # 查找所有CSV文件
    csv_files = list(data_dir.glob(pattern))
    print(f"找到 {len(csv_files)} 个CSV文件")
    
    if not csv_files:
        raise ValueError(f"在目录 {data_dir} 中未找到匹配 {pattern} 的文件")
    
    # 提取时间信息并排序
    time_info = []
    for csv_file in csv_files:
        filename = csv_file.stem
        time_match = re.search(r'(\d{6})', filename)  # 匹配YYYYMM格式
        if time_match:
            time_str = time_match.group(1)
            year = int(time_str[:4])
            month = int(time_str[4:6])
            time_info.append((year, month, csv_file))
    
    # 按时间排序
    time_info.sort(key=lambda x: (x[0], x[1]))
    
    print(f"时间范围: {time_info[0][0]:04d}-{time_info[0][1]:02d} 到 {time_info[-1][0]:04d}-{time_info[-1][1]:02d}")
    print(f"共 {len(time_info)} 个时间步")
    
    # 读取第一个文件来确定网格结构
    print("\n正在分析第一个文件以确定网格结构...")
    first_file = time_info[0][2]
    df_first = pd.read_csv(first_file)
    df_first = df_first.dropna(subset=['latitude', 'longitude', 'albedo'])
    
    print(f"第一个文件数据点数: {len(df_first)}")
    print(f"经度范围: {df_first['longitude'].min():.2f} 到 {df_first['longitude'].max():.2f}")
    print(f"纬度范围: {df_first['latitude'].min():.2f} 到 {df_first['latitude'].max():.2f}")
    
    # 创建统一的网格（使用所有文件的最大范围）
    resolution = 0.1
    all_lons, all_lats = [], []
    
    print("\n正在扫描所有文件以确定完整的空间范围...")
    for i, (year, month, csv_file) in enumerate(time_info):
        if i % 50 == 0:  # 每50个文件显示一次进度
            print(f"扫描进度: {i+1}/{len(time_info)}")
        
        df = pd.read_csv(csv_file)
        df = df.dropna(subset=['latitude', 'longitude', 'albedo'])
        all_lons.extend(df['longitude'].values)
        all_lats.extend(df['latitude'].values)
    
    # 确定完整的经纬度范围
    lon_min = np.floor(np.min(all_lons) / resolution) * resolution
    lon_max = np.ceil(np.max(all_lons) / resolution) * resolution
    lat_min = np.floor(np.min(all_lats) / resolution) * resolution
    lat_max = np.ceil(np.max(all_lats) / resolution) * resolution
    
    # 创建坐标数组
    lons = np.arange(lon_min, lon_max + resolution, resolution)
    lats = np.arange(lat_min, lat_max + resolution, resolution)
    
    print(f"最终网格大小: {len(lons)} x {len(lats)}")
    print(f"经度范围: {lon_min:.1f} 到 {lon_max:.1f}")
    print(f"纬度范围: {lat_min:.1f} 到 {lat_max:.1f}")
    
    # 创建时间数组
    times = []
    base_date = datetime(1950, 1, 1)
    for year, month, _ in time_info:
        target_date = datetime(year, month, 1)
        days_since = (target_date - base_date).days
        times.append(days_since)
    
    # 创建NetCDF文件
    print(f"\n正在创建NetCDF文件: {output_path}")
    
    with nc.Dataset(output_path, 'w', format='NETCDF4') as ncfile:
        # 创建维度
        ncfile.createDimension('time', len(time_info))
        ncfile.createDimension('lat', len(lats))
        ncfile.createDimension('lon', len(lons))
        
        # 创建坐标变量
        time_var = ncfile.createVariable('time', 'f8', ('time',))
        lat_var = ncfile.createVariable('lat', 'f8', ('lat',))
        lon_var = ncfile.createVariable('lon', 'f8', ('lon',))
        
        # 创建数据变量
        albedo_var = ncfile.createVariable('albedo', 'f4', ('time', 'lat', 'lon'), 
                                          fill_value=np.nan, zlib=True, complevel=4)
        
        # 填充坐标数据
        time_var[:] = times
        lat_var[:] = lats
        lon_var[:] = lons
        
        # 设置坐标变量属性
        time_var.units = 'days since 1950-01-01 00:00:00'
        time_var.calendar = 'gregorian'
        time_var.long_name = 'time'
        time_var.standard_name = 'time'
        
        lat_var.units = 'degrees_north'
        lat_var.long_name = 'latitude'
        lat_var.standard_name = 'latitude'
        
        lon_var.units = 'degrees_east'
        lon_var.long_name = 'longitude'
        lon_var.standard_name = 'longitude'
        
        albedo_var.units = 'dimensionless'
        albedo_var.long_name = 'surface albedo'
        albedo_var.standard_name = 'surface_albedo'
        albedo_var.valid_range = np.array([0.0, 1.0])
        
        # 逐个处理每个时间步的数据
        print("\n正在处理各时间步数据...")
        total_valid_points = 0
        
        for t, (year, month, csv_file) in enumerate(time_info):
            if t % 10 == 0:  # 每10个文件显示一次进度
                print(f"处理进度: {t+1}/{len(time_info)} ({year}-{month:02d})")
            
            # 读取当前文件
            df = pd.read_csv(csv_file)
            df = df.dropna(subset=['latitude', 'longitude', 'albedo'])
            
            # 初始化当前时间步的网格
            albedo_grid = np.full((len(lats), len(lons)), np.nan)
            
            # 将数据映射到网格
            valid_points = 0
            for _, row in df.iterrows():
                # 找到最近的网格点
                lon_idx = np.argmin(np.abs(lons - row['longitude']))
                lat_idx = np.argmin(np.abs(lats - row['latitude']))
                albedo_grid[lat_idx, lon_idx] = row['albedo']
                valid_points += 1
            
            # 填充到NetCDF变量
            albedo_var[t, :, :] = albedo_grid
            total_valid_points += valid_points
        
        # 设置全局属性
        ncfile.title = 'Greenland Surface Albedo Time Series'
        ncfile.source = f'Converted from CSV files in {data_dir.name}'
        ncfile.history = f'Created on {datetime.now().isoformat()}'
        ncfile.institution = 'TU Berlin'
        ncfile.references = 'Greenland albedo project'
        ncfile.Conventions = 'CF-1.6'
        ncfile.spatial_resolution = f'{resolution} degrees'
        ncfile.temporal_resolution = 'monthly'
        
        # 时间范围
        ncfile.time_coverage_start = f'{time_info[0][0]:04d}-{time_info[0][1]:02d}-01'
        ncfile.time_coverage_end = f'{time_info[-1][0]:04d}-{time_info[-1][1]:02d}-01'
        ncfile.time_coverage_duration = f'{len(time_info)} months'
        
        # 地理范围
        ncfile.geospatial_lat_min = float(lats.min())
        ncfile.geospatial_lat_max = float(lats.max())
        ncfile.geospatial_lon_min = float(lons.min())
        ncfile.geospatial_lon_max = float(lons.max())
        ncfile.geospatial_lat_resolution = resolution
        ncfile.geospatial_lon_resolution = resolution
        
        # 数据统计
        ncfile.number_of_time_steps = len(time_info)
        ncfile.total_data_points = total_valid_points
    
    print(f"\n合并完成!")
    print(f"输出文件: {output_path}")
    print(f"文件大小: {output_path.stat().st_size / (1024*1024):.1f} MB")
    print(f"时间步数: {len(time_info)}")
    print(f"网格大小: {len(lons)} × {len(lats)}")
    print(f"总数据点数: {total_valid_points}")
    
    return str(output_path)

def verify_netcdf_file(nc_file_path):
    """验证生成的NetCDF文件"""
    print(f"\n验证NetCDF文件: {nc_file_path}")
    
    with nc.Dataset(nc_file_path, 'r') as ncfile:
        print(f"维度: {dict(ncfile.dimensions)}")
        print(f"变量: {list(ncfile.variables.keys())}")
        
        # 时间信息
        time_var = ncfile.variables['time']
        print(f"时间范围: {len(time_var)} 个月")
        
        # 数据统计
        albedo_data = ncfile.variables['albedo']
        print(f"反照率数据形状: {albedo_data.shape}")
        
        # 计算每个时间步的有效数据点
        valid_counts = []
        for t in range(albedo_data.shape[0]):
            valid_count = np.sum(~np.isnan(albedo_data[t, :, :]))
            valid_counts.append(valid_count)
        
        print(f"平均每月有效数据点: {np.mean(valid_counts):.0f}")
        print(f"有效数据点范围: {np.min(valid_counts)} - {np.max(valid_counts)}")
        
        # 全局属性
        print(f"全局属性数量: {len(ncfile.ncattrs())}")
        if hasattr(ncfile, 'time_coverage_start'):
            print(f"时间覆盖: {ncfile.time_coverage_start} 到 {ncfile.time_coverage_end}")

# 使用示例
if __name__ == "__main__":
    # 你的数据目录路径
    data_directory = r"F:\TUB\greenlandProject\ne_10m_admin_0_countries\greenlandAlbedo\1995prediction"
    
    try:
        # 批量转换
        output_file = batch_csv_to_netcdf(
            data_dir=data_directory,
            output_filename='Greenland_albedo1950-2024.nc',
            pattern='*albedo.csv'
        )
        
        # 验证文件
        verify_netcdf_file(output_file)
        
    except Exception as e:
        print(f"处理失败: {e}")
        import traceback
        traceback.print_exc()

找到 450 个CSV文件
时间范围: 1950-04 到 2024-09
共 450 个时间步

正在分析第一个文件以确定网格结构...
第一个文件数据点数: 65472
经度范围: -72.70 到 -11.70
纬度范围: 59.90 到 83.50

正在扫描所有文件以确定完整的空间范围...
扫描进度: 1/450
扫描进度: 51/450
扫描进度: 101/450
扫描进度: 151/450
扫描进度: 201/450
扫描进度: 251/450
扫描进度: 301/450
扫描进度: 351/450
扫描进度: 401/450
最终网格大小: 612 x 237
经度范围: -72.7 到 -11.6
纬度范围: 59.9 到 83.5

正在创建NetCDF文件: F:\TUB\greenlandProject\ne_10m_admin_0_countries\greenlandAlbedo\1995prediction\Greenland_albedo1950-2024.nc

正在处理各时间步数据...
处理进度: 1/450 (1950-04)
处理进度: 11/450 (1951-08)
处理进度: 21/450 (1953-06)
处理进度: 31/450 (1955-04)
处理进度: 41/450 (1956-08)
处理进度: 51/450 (1958-06)
处理进度: 61/450 (1960-04)
处理进度: 71/450 (1961-08)
处理进度: 81/450 (1963-06)
处理进度: 91/450 (1965-04)
处理进度: 101/450 (1966-08)
处理进度: 111/450 (1968-06)
处理进度: 121/450 (1970-04)
处理进度: 131/450 (1971-08)
处理进度: 141/450 (1973-06)
处理进度: 151/450 (1975-04)
处理进度: 161/450 (1976-08)
处理进度: 171/450 (1978-06)
处理进度: 181/450 (1980-04)
处理进度: 191/450 (1981-08)
处理进度: 201/450 (1983-06)
处理进度: 211/450 (1985-04)
处理进度: 221/450

In [ ]:
代码省略：  用shp得到greenland的F:\TUB\greenlandProject\ERA5LAND\ProcessFile\greenland_grid_points.csv（存储0.1*0.1经纬度网格）

In [2]:
# -*- coding: utf-8 -*-
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc


# 近邻格点索引（四舍五入），避免银行家舍入
def _to_idx(vals, vmin, res, nmax):
    idx = np.floor((vals - vmin) / res + 0.5).astype(np.int64)
    return np.clip(idx, 0, nmax - 1)


def build_albedo_netcdf(
    data_dir=r"C:\greenland\NosdePrediction",
    pattern="*prediction.csv",
    out_name="greenlandNoSde.nc",
    res=0.1
):
    data_dir = Path(data_dir)
    files = sorted(data_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"未找到 {data_dir}\\{pattern}")

    # 解析 YYYYMM 与过滤无效文件
    ym_fps = []
    for f in files:
        m = re.search(r"(\d{6})", f.stem)
        if m:
            ym = m.group(1)
            ym_fps.append((int(ym[:4]), int(ym[4:6]), f))
    if not ym_fps:
        raise RuntimeError("文件名未解析到 YYYYMM")

    ym_fps.sort(key=lambda x: (x[0], x[1]))
    years_months = [(y, m) for y, m, _ in ym_fps]

    # 一次扫描确定 lon/lat 范围（仅读两列，快）
    lon_min = lon_max = lat_min = lat_max = None
    for _, _, f in ym_fps:
        df = pd.read_csv(f, usecols=["longitude", "latitude"], dtype="float32").dropna()
        if df.empty:
            continue
        lon_min = float(df.longitude.min()) if lon_min is None else min(lon_min, float(df.longitude.min()))
        lon_max = float(df.longitude.max()) if lon_max is None else max(lon_max, float(df.longitude.max()))
        lat_min = float(df.latitude.min()) if lat_min is None else min(lat_min, float(df.latitude.min()))
        lat_max = float(df.latitude.max()) if lat_max is None else max(lat_max, float(df.latitude.max()))

    if None in (lon_min, lon_max, lat_min, lat_max):
        raise RuntimeError("无法确定经纬度范围（文件可能为空）")

    # 边界微扩，生成规则网格
    eps = 1e-6
    lon_min = np.floor((lon_min - eps) / res) * res
    lon_max = np.ceil((lon_max + eps) / res) * res
    lat_min = np.floor((lat_min - eps) / res) * res
    lat_max = np.ceil((lat_max + eps) / res) * res

    lons = np.arange(lon_min, lon_max + res, res, dtype="float32")
    lats = np.arange(lat_min, lat_max + res, res, dtype="float32")
    nlon, nlat, nt = lons.size, lats.size, len(ym_fps)

    # 时间轴（days since 1950-01-01）
    base = datetime(1950, 1, 1)
    times = np.array([(datetime(y, m, 1) - base).days for y, m in years_months], dtype="int32")

    # 写 NetCDF（压缩、shuffle、合理分块）
    out_path = data_dir / out_name
    with nc.Dataset(out_path, "w", format="NETCDF4") as ds:
        ds.createDimension("time", nt)
        ds.createDimension("lat", nlat)
        ds.createDimension("lon", nlon)

        tvar = ds.createVariable("time", "i4", ("time",))
        yvar = ds.createVariable("lat", "f4", ("lat",))
        xvar = ds.createVariable("lon", "f4", ("lon",))
        aval = ds.createVariable(
            "albedo", "f4", ("time", "lat", "lon"),
            zlib=True, complevel=4, shuffle=True,
            chunksizes=(1, min(256, nlat), min(256, nlon)),
            fill_value=np.nan
        )

        tvar[:] = times
        yvar[:] = lats
        xvar[:] = lons

        tvar.units = "days since 1950-01-01 00:00:00"; tvar.calendar = "gregorian"
        yvar.units = "degrees_north"; yvar.standard_name = "latitude"
        xvar.units = "degrees_east";  xvar.standard_name = "longitude"
        aval.units = "1"; aval.long_name = "surface albedo"; aval.standard_name = "surface_albedo"
        aval.valid_range = np.array([0.0, 1.0], dtype="f4")

        for ti, (y, m, f) in enumerate(ym_fps):
            df = pd.read_csv(f, usecols=["longitude", "latitude", "albedo"], dtype="float32").dropna()
            # 单值约束
            df = df[(df.albedo >= 0.0) & (df.albedo <= 1.0)]
            if df.empty:
                aval[ti, :, :] = np.nan
                continue

            lon = df.longitude.to_numpy(copy=False)
            lat = df.latitude.to_numpy(copy=False)
            alb = df.albedo.to_numpy(copy=False)

            xi = _to_idx(lon, lon_min, res, nlon)
            yi = _to_idx(lat, lat_min, res, nlat)

            # 写入网格（假设同格点唯一；若偶有重复，取最后一条）
            grid = np.full((nlat, nlon), np.nan, dtype="float32")
            grid[yi, xi] = alb
            aval[ti, :, :] = grid

        # 元数据
        ds.title = "Greenland Surface Albedo Time Series (No SDE)"
        ds.source = f"CSV to NetCDF from {data_dir.as_posix()}"
        ds.history = f"Created on {datetime.now().isoformat()}"
        ds.institution = "TU Berlin"
        ds.Conventions = "CF-1.6"
        ds.spatial_resolution = f"{res} degrees"
        ds.temporal_resolution = "monthly"
        ds.time_coverage_start = f"{years_months[0][0]:04d}-{years_months[0][1]:02d}-01"
        ds.time_coverage_end   = f"{years_months[-1][0]:04d}-{years_months[-1][1]:02d}-01"

    print(f"Done: {out_path}")
    return str(out_path)


if __name__ == "__main__":
    build_albedo_netcdf()


Done: C:\greenland\NosdePrediction\greenlandNoSde.nc


In [8]:
# -*- coding: utf-8 -*-
import xarray as xr
import geopandas as gpd
import regionmask
from pathlib import Path

# 输入文件
nc_path = Path(r"C:\greenland\NosdePrediction\greenlandNoSde.nc")
# 输出文件
out_path = nc_path.with_name(nc_path.stem + "_clip.nc")
# shapefile
shp_path = Path(r"C:\greenland\Greenland_shp\Gboundry.shp")

# 1. 打开 NetCDF
ds = xr.open_dataset(nc_path)

# 2. 读取格陵兰边界
gdf = gpd.read_file(shp_path)
if gdf.crs is not None and gdf.crs.to_string().lower() != "epsg:4326":
    gdf = gdf.to_crs("EPSG:4326")  # 确保与nc经纬度一致

# 3. 构建掩膜 (regionmask)
mask = regionmask.mask_geopandas(gdf, ds["lon"], ds["lat"])

# 4. 应用掩膜：边界外 → NaN
ds_clip = ds.where(mask == 0)  # gdf只有一个多边形，mask==0 表示在边界内

# 5. 保存新文件
ds_clip.to_netcdf(out_path)

print(f"已裁剪并保存: {out_path}")


已裁剪并保存: C:\greenland\NosdePrediction\greenlandNoSde_clip.nc


In [2]:
import xarray as xr

# --- 文件路径和变量设置 ---
# 输入文件路径
input_file = r"C:\greenland\1995bu\Greenland_albedo1950-2024.nc"

# 输出文件路径
output_file = r"C:\greenland\1995bu\Greenland_albedo1950-2024Summer.nc"

# 要处理的变量名称（根据你的 NetCDF 文件实际情况修改，这里假设是 'alb'）
variable_name = 'albedo'

# --- 1. 读取数据 ---
try:
    ds = xr.open_dataset(input_file)
    print(f"成功打开文件: {input_file}")
except FileNotFoundError:
    print(f"错误：找不到文件 {input_file}")
    exit()
except Exception as e:
    print(f"打开文件时发生错误: {e}")
    exit()

# 确保 'time' 维度是 datetime 类型，以便提取月份
if 'time' not in ds.coords and 'time' not in ds.dims:
    print("错误：数据集中没有名为 'time' 的时间维度。")
    exit()

# --- 2. 筛选夏季月份 (6, 7, 8) ---
# 使用 .sel() 结合时间访问器 .dt.month.isin() 来选择指定月份的数据
summer_ds = ds.sel(time=ds.time.dt.month.isin([6, 7, 8]))
print(f"已筛选出 6、7、8 月的数据。总时间步长: {len(summer_ds['time'])}")

# 检查筛选后是否有数据
if len(summer_ds['time']) == 0:
    print("警告：筛选后没有找到 6, 7, 8 月份的数据。请检查时间维度的值。")
    # 尝试检查变量是否存在
    if variable_name not in ds.data_vars:
        print(f"错误：变量 '{variable_name}' 不存在于数据集中。请修改 variable_name。")
    exit()

# --- 3. 计算每年的夏季均值 ---
# 使用 .groupby('time.year') 按年分组
# 然后对时间维度 'time' 求平均 .mean('time')
# 默认情况下，xarray 在计算季度或季节平均值时是**等权重**平均（即每个月权重相同）。
# 对于月平均数据求季节平均，通常采用等权重平均是可接受的。
# 如果数据是日平均数据，推荐使用 xarray 的日权重方法（比较复杂，此处省略，如果你需要可以再问我）。

# 选择变量并计算年平均
summer_mean_da = summer_ds[variable_name].groupby('time.year').mean('time')

# 将 DataArray 转换回 Dataset
summer_mean_ds = summer_mean_da.to_dataset(name=variable_name)

print("已计算出每年的夏季均值 (6、7、8月)。")
print("结果 Dataset 概览:")
print(summer_mean_ds)

# --- 4. 保存到新的 NetCDF 文件 ---
try:
    # 重新命名时间坐标为 'year'，并修改其属性以反映这是年平均数据
    summer_mean_ds = summer_mean_ds.rename({'year': 'time'})
    summer_mean_ds['time'].attrs['long_name'] = 'Annual Summer Mean Year'
    summer_mean_ds[variable_name].attrs['standard_name'] = f'{variable_name}_annual_summer_mean'

    summer_mean_ds.to_netcdf(output_file)
    print(f"\n成功保存结果到新文件: {output_file}")
except Exception as e:
    print(f"\n保存文件时发生错误: {e}")

成功打开文件: C:\greenland\1995bu\Greenland_albedo1950-2024.nc
已筛选出 6、7、8 月的数据。总时间步长: 225
已计算出每年的夏季均值 (6、7、8月)。
结果 Dataset 概览:
<xarray.Dataset> Size: 44MB
Dimensions:  (lat: 237, lon: 612, year: 75)
Coordinates:
  * lat      (lat) float64 2kB 59.9 60.0 60.1 60.2 60.3 ... 83.2 83.3 83.4 83.5
  * lon      (lon) float64 5kB -72.7 -72.6 -72.5 -72.4 ... -11.8 -11.7 -11.6
  * year     (year) int64 600B 1950 1951 1952 1953 1954 ... 2021 2022 2023 2024
Data variables:
    albedo   (year, lat, lon) float32 44MB nan nan nan nan ... nan nan nan nan

成功保存结果到新文件: C:\greenland\1995bu\Greenland_albedo1950-2024Summer.nc


In [1]:
import xarray as xr
import pandas as pd

# 文件路径
file_path = "C:\\Users\\greenlandProject\\ERA5LAND\\ProcessFile\\ERA5LANDfeature_greenland.nc"

print(f"正在读取文件: {file_path}")

try:
    # 使用 xarray 打开 NetCDF 文件
    ds = xr.open_dataset(file_path)

    print("\n--- 成功读取文件 ---")

    # 1. 打印数据集概览 (Dimensions, Coordinates, Variables, Attributes)
    print("\n### 数据集概览 (ds):")
    print(ds)

    # 2. 打印所有变量名
    print("\n### 所有变量名:")
    for var in ds.data_vars:
        print(f"- {var} (Units: {ds[var].attrs.get('units', 'N/A')})")

    # 3. 检查时间维度
    # 假设时间维度名为 'time'，如果不是，你需要将其改为正确的名称
    time_dim_name = 'time'

    if time_dim_name in ds.coords:
        time_coords = ds[time_dim_name]
        print(f"\n### 时间坐标 ({time_dim_name}) 信息:")
        print(f" - 数据类型 (dtype): {time_coords.dtype}")
        print(f" - 前5个时间值: {time_coords.values[:5]}")
        print(f" - 时间范围: 从 {time_coords.values[0]} 到 {time_coords.values[-1]}")

        # 检查时间数据类型是否为 datetime64
        if pd.api.types.is_datetime64_any_dtype(time_coords):
            print(" - **时间格式确认: 已经是 datetime64 类型，可直接进行时间序列操作。**")
        else:
            print(" - **注意: 时间数据类型不是标准的 datetime64。**")
            print(" - 如果后续计算报错，可能需要显式转换，例如：`ds['time'] = pd.to_datetime(ds['time'].values)`")

    else:
        print(f"\n**警告: 数据集中没有找到名为 '{time_dim_name}' 的时间坐标。**")
        print("请检查数据集概览 (ds) 中的坐标 (Coordinates) 部分，找到正确的时间维度名称。")

    # 4. 示例：选择一个变量进行操作
    # 假设你想要处理 't2m' (2米温度) 这个变量
    if 't2m' in ds.data_vars:
        var_to_process = 't2m'
        print(f"\n### 示例变量 '{var_to_process}' 的形状 (Shape):")
        print(ds[var_to_process].shape)
    else:
        print("\n***找不到示例变量 't2m'，请使用你数据中实际的变量名。***")

except FileNotFoundError:
    print(f"\n错误: 文件未找到，请检查路径是否正确: {file_path}")
except Exception as e:
    print(f"\n读取文件时发生错误: {e}")

正在读取文件: C:\Users\greenlandProject\ERA5LAND\ProcessFile\ERA5LANDfeature_greenland.nc

--- 成功读取文件 ---

### 数据集概览 (ds):
<xarray.Dataset> Size: 5GB
Dimensions:      (valid_time: 902, latitude: 239, longitude: 612)
Coordinates:
    number       int64 8B ...
  * valid_time   (valid_time) datetime64[ns] 7kB 1950-01-01 ... 2025-02-01
  * latitude     (latitude) float64 2kB 83.6 83.5 83.4 83.3 ... 60.0 59.9 59.8
  * longitude    (longitude) float64 5kB -72.7 -72.6 -72.5 ... -11.8 -11.7 -11.6
    expver       (valid_time) <U4 14kB ...
Data variables:
    spatial_ref  int64 8B ...
    t2m          (valid_time, latitude, longitude) float32 528MB ...
    asn          (valid_time, latitude, longitude) float32 528MB ...
    fal          (valid_time, latitude, longitude) float32 528MB ...
    slhf         (valid_time, latitude, longitude) float32 528MB ...
    str          (valid_time, latitude, longitude) float32 528MB ...
    ssrd         (valid_time, latitude, longitude) float32 528MB ...
    strd 

In [3]:
import xarray as xr
import pandas as pd

# ----------------- 配置 -----------------
# 文件路径
file_path = "C:\\Users\\greenlandProject\\ERA5LAND\\ProcessFile\\ERA5LANDfeature_greenland.nc"

# 要处理的所有变量名列表
variable_names = [
    't2m', 'asn', 'fal', 'slhf', 'str', 'ssrd', 'strd', 'u10', 'v10', 'sp'
]

# 输出 CSV 文件路径 (新文件)
output_csv_path = "C:\\Users\\11823\\era5land\\era5_Environ_feature_timeseries.csv"

# 时间维度名称
time_dim_name = 'valid_time'

# ----------------- 代码实现 -----------------

print(f"正在读取文件: {file_path}")

try:
    # 1. 使用 xarray 打开 NetCDF 文件
    ds = xr.open_dataset(file_path)
    
    # 2. 筛选出夏季 (JJA: June, July, August) 的数据
    # 只进行一次时间筛选，提高效率
    jja_data = ds.sel({time_dim_name: ds[time_dim_name].dt.month.isin([6, 7, 8])})

    print(f"\n已筛选出夏季 (JJA) 数据。")
    print(f"JJA数据时间范围: {jja_data[time_dim_name].values[0]} - {jja_data[time_dim_name].values[-1]}")
    
    # 初始化一个空的 DataFrame 来存储所有变量的结果
    all_vars_df = pd.DataFrame()

    # 3. 循环计算每个变量的每年夏季全域均值
    print("\n--- 正在计算并合并各变量的夏季均值 ---")
    
    for var in variable_names:
        if var in jja_data.data_vars:
            print(f" -> 正在处理变量: {var}")
            
            # a. 对空间维度 (latitude, longitude) 求平均
            # 得到一个仅有时间维度的 DataArray
            spatial_mean = jja_data[var].mean(dim=('latitude', 'longitude'))
            
            # b. 按年份分组并求时间平均 (每年JJA的平均)
            # 结果是一个以 'year' 为坐标轴的 DataArray
            jja_yearly_mean = spatial_mean.groupby(spatial_mean[time_dim_name].dt.year).mean()
            
            # c. 转换为 Pandas Series
            jja_series = jja_yearly_mean.to_series()
            
            # d. 设置列名
            jja_series.name = f'{var}_JJA_Mean'
            
            # e. 合并到主 DataFrame 中
            if all_vars_df.empty:
                all_vars_df = pd.DataFrame(jja_series)
            else:
                all_vars_df = all_vars_df.join(jja_series, how='outer')
        else:
            print(f"警告: 数据集中未找到变量 '{var}'，已跳过。")


    # 4. 数据清理和格式化
    
    # 将索引名设置为“年份”
    all_vars_df.index.name = '年份'
    
    # 移除不完整的年份 (假设你的数据到 2025 年 2 月，则 2025 年的 JJA 数据不完整)
    if 2025 in all_vars_df.index:
        all_vars_df = all_vars_df.drop(2025)
        print("\n**注意：已移除不完整的 2025 年数据，最终时间范围为 1950-2024。**")
    
    # 5. 写入 CSV 文件
    
    # 打印前5行数据以供检查
    print("\n### 准备写入 CSV 的数据 (前5行):")
    print(all_vars_df.head())
    print(f"\n数据范围: {all_vars_df.index.min()} - {all_vars_df.index.max()}")

    # 写入文件，index=True 将 '年份' 列保留
    all_vars_df.to_csv(output_csv_path, encoding='utf-8')

    print(f"\n--- 成功写入文件 ---")
    print(f"数据已保存到: {output_csv_path}")
    print(f"CSV 中包含 {len(all_vars_df)} 条记录 ({all_vars_df.index.min()} 到 {all_vars_df.index.max()})。")


except FileNotFoundError:
    print(f"\n错误: 文件未找到，请检查路径是否正确: {file_path}")
except Exception as e:
    print(f"\n处理文件时发生错误: {e}")

正在读取文件: C:\Users\greenlandProject\ERA5LAND\ProcessFile\ERA5LANDfeature_greenland.nc

已筛选出夏季 (JJA) 数据。
JJA数据时间范围: 1950-06-01T00:00:00.000000000 - 2024-08-01T00:00:00.000000000

--- 正在计算并合并各变量的夏季均值 ---
 -> 正在处理变量: t2m
 -> 正在处理变量: asn
 -> 正在处理变量: fal
 -> 正在处理变量: slhf
 -> 正在处理变量: str
 -> 正在处理变量: ssrd
 -> 正在处理变量: strd
 -> 正在处理变量: u10
 -> 正在处理变量: v10
 -> 正在处理变量: sp

### 准备写入 CSV 的数据 (前5行):
      t2m_JJA_Mean  asn_JJA_Mean  fal_JJA_Mean  slhf_JJA_Mean  str_JJA_Mean  \
年份                                                                            
1950    266.535187      0.834076      0.803438  -312743.93750   -4619406.50   
1951    265.976532      0.832140      0.803186  -288577.00000   -4344184.50   
1952    265.690033      0.834470      0.802447  -330634.96875   -4424421.50   
1953    266.141937      0.833577      0.807537  -334766.65625   -4410751.50   
1954    266.203094      0.832684      0.805197  -344933.65625   -4054123.75   

      ssrd_JJA_Mean  strd_JJA_Mean  u10_JJA_Mean  v10_JJA_M

In [5]:
import xarray as xr
import rioxarray # 激活 rioxarray 扩展
import geopandas as gpd
import os
import sys

# --- 定义文件路径 ---
nc_file_path = "C:\\Users\\greenlandProject\\ERA5LAND\\ProcessFile\\snow3para.nc"
shp_file_path = "C:\\greenland\\Greenland_shp\\流域冰盖\\new\\Gboundry.shp"

# 构造输出文件名
base_name = os.path.splitext(os.path.basename(nc_file_path))[0]
output_dir = os.path.dirname(nc_file_path)
output_file_name = f"{base_name}_greenland.nc"
output_file_path = os.path.join(output_dir, output_file_name)

# 初始化裁剪后的数据集变量
clipped_ds = None

# --- 1. 读取和准备数据 (基于诊断结果修复) ---
print(f"--- 正在加载数据和Shapefile ---")
try:
    # 1.1 读取 Shapefile 获取目标 CRS
    gdf = gpd.read_file(shp_file_path)
    target_crs = gdf.crs # 目标 CRS 默认为 EPSG:4326 (WGS84)
    print(f"Shapefile 的目标 CRS: {target_crs}")
    
    # 1.2 读取 NetCDF 文件
    ds = xr.open_dataset(nc_file_path, chunks='auto')

    # 1.3 【核心修复】将经度坐标从 0-360 转换为 -180 到 180
    # 这一步使用 xarray 提供的标准方法，比手动计算更可靠
    ds['longitude'] = (ds['longitude'] + 180) % 360 - 180
    
    # 排序经度，以确保裁剪能够正常工作（可能导致数据重排，但对裁剪必须）
    ds = ds.sortby(ds.longitude)
    
    print("经度已从 0-360 转换为 -180 到 180 范围，并已排序。")
    print(f"转换后的经度范围: {ds['longitude'].values[0]:.2f} 到 {ds['longitude'].values[-1]:.2f}")
    
    # 1.4 设置空间维度和 CRS (您的文件已包含正确的维度名)
    ds = ds.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude', inplace=True)
    
    # 根据文件信息，CRS 应该已经是 WGS84 (EPSG:4326)，但我们再写一次确保 rioxarray 识别
    ds.rio.write_crs("EPSG:4326", inplace=True)
    print("空间维度和 CRS (EPSG:4326) 设置完成。")

    # Shapefile 目标 CRS 已经是 EPSG:4326，所以无需重投影。

except Exception as e:
    print(f"\n读取或准备数据时发生错误: {e}")
    sys.exit(1)


# --- 2. 使用 Shapefile 进行裁剪 ---
print(f"--- 正在使用 Shapefile: {shp_file_path} 进行裁剪 ---")
try:
    # 使用 rioxarray 的 clip 方法进行裁剪
    clipped_ds = ds.rio.clip(
        geometries=gdf.geometry,
        crs=target_crs,
        drop=True,
        invert=False,
        all_touched=True
    )
    
    # 检查裁剪后的数据集是否为空
    if not clipped_ds or clipped_ds.sizes['longitude'] == 0:
        raise RuntimeError("裁剪结果为空。请检查Shapefile与数据范围是否重叠。")
        
    print("裁剪成功。")

except Exception as e:
    print(f"\n裁剪过程中发生错误: {e}")
    clipped_ds = None


# --- 3. 保存新的 NetCDF 文件 ---
if clipped_ds is not None:
    print(f"--- 正在将裁剪后的数据保存到: {output_file_path} ---")
    try:
        # 只保留需要的变量
        final_ds = clipped_ds[['snowc', 'sde']]
        
        # 保存裁剪后的数据集为新的 NetCDF 文件
        final_ds.to_netcdf(output_file_path, mode='w')
        print("---------------------------------------")
        print(f"成功！新的 NetCDF 文件已保存为: {output_file_path}")

    except Exception as e:
        print(f"保存文件时发生错误: {e}")
else:
    print("\n由于裁剪失败或结果为空，跳过文件保存步骤。")

--- 正在加载数据和Shapefile ---
Shapefile 的目标 CRS: EPSG:4326
经度已从 0-360 转换为 -180 到 180 范围，并已排序。
转换后的经度范围: -80.00 到 -5.00
空间维度和 CRS (EPSG:4326) 设置完成。
--- 正在使用 Shapefile: C:\greenland\Greenland_shp\流域冰盖\new\Gboundry.shp 进行裁剪 ---
裁剪成功。
--- 正在将裁剪后的数据保存到: C:\Users\greenlandProject\ERA5LAND\ProcessFile\snow3para_greenland.nc ---
---------------------------------------
成功！新的 NetCDF 文件已保存为: C:\Users\greenlandProject\ERA5LAND\ProcessFile\snow3para_greenland.nc


In [5]:
import os
import rasterio
import numpy as np

# 1. 定义路径
# 请确保将这里的路径替换为您的实际路径
TIFF_FOLDER_PATH = r"C:\app\TUB\视域分析demo_月球南极永久阴影区"

def analyze_tiff_metadata(folder_path):
    """
    遍历指定文件夹，读取并解析所有TIFF文件的坐标系、分辨率和数据含义。
    """
    print(f"--- 开始分析文件夹: {folder_path} ---")
    
    # 初始化结果列表，用于汇总信息
    results = []

    # 遍历文件夹中的所有文件
    for filename in os.listdir(folder_path):
        if filename.endswith(('.tif', '.tiff')):
            file_path = os.path.join(folder_path, filename)
            
            # 初始化当前文件的解析字典
            file_info = {
                "文件名": filename,
                "含义推断": "未知",
                "坐标系 (CRS)": "未定义",
                "数据类型": "N/A",
                "分辨率 (X, Y)": "N/A",
                "时间信息": "未直接存储" # 默认时间信息为未直接存储
            }

            print(f"\n==================================================")
            print(f"正在处理文件: {filename}")
            
            try:
                with rasterio.open(file_path) as src:
                    
                    # --- 1. 坐标系 (CRS) ---
                    crs_info = src.crs.to_string() if src.crs else '未定义'
                    file_info["坐标系 (CRS)"] = crs_info
                    print(f"  > 坐标系 (CRS): {crs_info}")

                    # --- 2. 数据分辨率 ---
                    # Transform矩阵包含分辨率信息
                    x_res = abs(src.transform.a)
                    y_res = abs(src.transform.e)
                    
                    # 尝试推断单位：如果CRS包含"meter"或未定义，通常假定单位是米
                    unit_str = " (可能为米/像素)"
                    if src.crs and ('deg' in src.crs.to_string().lower() or 'latlong' in src.crs.to_string().lower()):
                         unit_str = " (度/像素)"
                    
                    res_info = f"X: {x_res:.3f}{unit_str}, Y: {y_res:.3f}{unit_str}"
                    file_info["分辨率 (X, Y)"] = res_info
                    print(f"  > 数据分辨率: {res_info}")
                    
                    # --- 3. 数据类型和统计 ---
                    data_dtype = src.dtypes[0] if src.count >= 1 else None
                    file_info["数据类型"] = str(data_dtype)
                    
                    if src.count >= 1:
                        data = src.read(1)
                        nodata_val = src.nodata if src.nodata is not None else src.nodatavals[0] if src.nodatavals else None
                        
                        if nodata_val is not None:
                            nodata_val_typed = np.array([nodata_val]).astype(data.dtype)[0]
                            valid_data = data[data != nodata_val_typed]
                        else:
                            valid_data = data
                        
                        if valid_data.size > 0:
                            print(f"  > 值范围: Min={np.min(valid_data):.2f}, Max={np.max(valid_data):.2f}")
                        else:
                            print("  > 警告: 发现有效数据点数量为零。")
                    
                    # --- 4. 含义推断 ---
                    filename_lower = filename.lower()
                    if 'dtm' in filename_lower or 'ldem' in filename_lower or 'elev' in filename_lower:
                        file_info["含义推断"] = "数字高程模型 (DEM/DTM)：表示月球表面的地形高度。"
                    elif 'psr' in filename_lower or 'shadow' in filename_lower:
                        file_info["含义推断"] = "永久阴影区 (PSR) 掩膜：二值数据，用于标记永久阴影区域（通常 1=阴影，0=非阴影）。"
                    elif 'sun' in filename_lower or 'light' in filename_lower or 'irrad' in filename_lower or 'illum' in filename_lower:
                        file_info["含义推断"] = "光照/日照数据：表示地表在特定时段内接收到的太阳辐射总量或光照时间。"
                    else:
                        file_info["含义推断"] = "未知：请查阅项目文档。"
                    
                    print(f"  > 含义推断: {file_info['含义推断']}")
                    
                    # --- 5. 时间信息 (Temporal Information) ---
                    # 尝试从元数据标签中寻找时间信息，通常不包含在标准GeoTIFF中
                    tags = src.tags(1) # 获取第一个波段的标签
                    datetime_tag = tags.get('DATETIME') or tags.get('TIFFTAG_DATETIME')
                    
                    if datetime_tag:
                         file_info["时间信息"] = f"从标签中提取: {datetime_tag}"
                    elif 'annual' in filename_lower or 'yearly' in filename_lower:
                         file_info["时间信息"] = "推测为年平均或年累计数据。"
                    else:
                        print("  > 时间信息: 标准元数据中未找到明确的时间标签。")
                    
                    results.append(file_info)

            except rasterio.RasterioIOError as e:
                print(f"  错误: 无法读取文件 {filename}。错误信息: {e}")
            except Exception as e:
                print(f"  发生意外错误: {e}")

    print("\n--- 所有文件处理完毕 ---")
    
    # 打印最终汇总表格
    print("\n\n##################################################")
    print("## TIFF 文件信息汇总表 ##")
    print("##################################################")
    
    # 格式化打印结果
    for info in results:
        print(f"\n文件: {info['文件名']}")
        print(f"  - 含义推断: {info['含义推断']}")
        print(f"  - 坐标系 (CRS): {info['坐标系 (CRS)']}")
        print(f"  - 数据分辨率: {info['分辨率 (X, Y)']}")
        print(f"  - 数据类型: {info['数据类型']}")
        print(f"  - 时间信息: {info['时间信息']}")

# 运行函数
analyze_tiff_metadata(TIFF_FOLDER_PATH)

--- 开始分析文件夹: C:\app\TUB\视域分析demo_月球南极永久阴影区 ---

正在处理文件: dtm.tif
  > 坐标系 (CRS): PROJCS["Moon2000_spole",GEOGCS["GCS_Moon",DATUM["Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],PRIMEM["Reference_Meridian",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-90],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",NORTH],AXIS["Northing",NORTH]]
  > 数据分辨率: X: 60.000 (度/像素), Y: 60.000 (度/像素)
  > 值范围: Min=nan, Max=nan
  > 含义推断: 数字高程模型 (DEM/DTM)：表示月球表面的地形高度。
  > 时间信息: 标准元数据中未找到明确的时间标签。

正在处理文件: ldem60m.tif
  > 坐标系 (CRS): PROJCS["Moon2000_spole",GEOGCS["GCS_Moon",DATUM["Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],PRIMEM["Reference_Meridian",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-90],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMET